In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage,它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role)创建了对应的BaseMessage的子类：

- SystemMessage: role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage: role是user，代表用户输入的消息
- AIMessage: role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage: role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    获取某个地方的天气
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent,发送消息
# response = agent.invoke({
#     "messages" : [
#         {"role": "system", "content": "你是一个热心的AI助手."},
#         {"role": "user", "content": "你好，我是Michale."},
#         {"role": "assistant", "content": "你好，Michale,很高兴认识你."},
#         {"role": "user", "content": "杭州今天天气如何？"}
#     ]
# })

# print(response)

response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是Michael"),
        AIMessage("你好，Michale，很高兴认识你。"),
        HumanMessage("杭州今天天气如何？")
    ]
})

print(response)

In [ ]:
for message in response['messages']:
    message.pretty_print()

# 多模态消息

之前我们都是向模型发送文本消息，但是LangChain也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：

- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。图片如下：
![示例图片](https://p0.ssl.qhimgs1.com/t04869400749fa927b0.jpg)

In [ ]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-plus", #这是一个支持图片、文本、视频多模态的模型
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [ ]:
# 创建Agent
agent = create_agent(model=model)

In [ ]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述一下这张图片"},
        {"type": "image", "url": "https://p0.ssl.qhimgs1.com/t04869400749fa927b0.jpg?"},
    ])

In [7]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张图片展示了一个温馨的室内场景，主要包含以下内容：

1.  **人物**：画面中央蹲着一位留着黑色短发的年轻人（看起来像是一位女性）。她身穿黑色的长袖上衣和一条膝盖处有破洞的蓝色牛仔裤。她的神情看起来很温柔，正低头看着身下的动物。

2.  **动物**：地上侧躺着一只体型硕大的金毛寻回犬（Golden Retriever）。它的毛色金黄蓬松，看起来非常放松和舒适，眼睛半闭，似乎很享受被抚摸的感觉。

3.  **动作与互动**：这个人正用双手轻柔地抚摸着狗狗的胸口和腹部区域。狗狗四脚朝天或者侧躺着，展现出对这个人完全的信任和依赖。

4.  **背景与环境**：
    *   地面是浅黄色的木地板。
    *   背景是一面白色的墙壁。
    *   值得注意的是，墙壁左侧和右侧有一些看起来像是被涂抹、模糊处理过的痕迹（可能是后期打码或者是墙上的污渍/涂鸦），右侧角落里似乎还有一些纸箱类的杂物。

总体来说，这是一张充满生活气息的照片，捕捉到了人与宠物之间亲密、宁静的互动时刻。

## 本地图片数据

有时候用户会上传图片数据，而不是图片的URL地址，我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。

```bash
uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='image/*', multiple=False)
display(uploader)


FileUpload(value=(), accept='image/*', description='Upload')

In [9]:
print(uploader.value)

()


In [10]:
# 读取图片，转换为base64字符串
import base64

# 方式一：使用第 13 单元的上传组件（必须先在组件里选图片，并点击 Upload 按钮）
if "uploader" not in globals() or not uploader.value:
    raise RuntimeError(
        "还没拿到图片：请先跑第 13 单元，在下方组件里选择图片并点击 Upload 按钮，再重跑本单元。"
        "如果组件点不动，就改用下面 方式二 的本地图片写法。"
    )

uploaded_file = uploader.value[0]
img_bytes = bytes(uploaded_file["content"])
img_b64 = base64.b64encode(img_bytes).decode("utf-8")
img_mime = uploaded_file["type"] or "image/jpeg"

# 方式二（备选）：直接读本地图片，跳过上传组件——注释掉上面三行，改用下面两行
# img_b64 = base64.b64encode(open(r"C:\Users\15893\Pictures\demo.jpg", "rb").read()).decode("utf-8")
# img_mime = "image/jpeg"

print(f"图片就绪：base64 长度 {len(img_b64)}，类型 {img_mime}")


RuntimeError: 还没拿到图片：请先跑第 13 单元，在下方组件里选择图片并点击 Upload 按钮，再重跑本单元。如果组件点不动，就改用下面 方式二 的本地图片写法。

In [ ]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": img_mime,
    },
    {"type": "text", "text": "给我讲一下图的内容是啥"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)